In [1]:
%pwd

'D:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model\\research'

# 1. Import the libraries:

In [16]:
import os
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                               GradientBoostingClassifier)
from xgboost import XGBClassifier
from sklearn.svm import SVC
from scipy.stats import loguniform, uniform
import matplotlib.pyplot as plt
import numpy as np
import time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib
import dagshub
from dotenv import load_dotenv
# os.chdir("..")

In [7]:
load_dotenv()

True

In [3]:
%pwd

'D:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model'

# 2. Configuration:

In [13]:
mlflow_tracking_uri = os.getenv("MLFLOW_URI")
dagshub_owner_name = os.getenv("REPO_OWNER")
dagshub_repo_name = os.getenv("REPO_NAME")
dagshub_user_token = os.getenv("DAGSHUB_USER_TOKEN")

In [14]:
mlflow.set_tracking_uri(mlflow_tracking_uri)
dagshub.init(repo_owner=dagshub_owner_name, repo_name=dagshub_repo_name, mlflow=True)

Initialized MLflow to track repo "Nishan-k/An-end-to-end-customer-churn-prediction-model"

Repository Nishan-k/An-end-to-end-customer-churn-prediction-model initialized!

# 3. Common functions:

In [18]:
def create_dep_indep_features(df:pd.DataFrame):
    """
    Splits the data into dependent(y) and independent(X) features:
    """
    df.dropna(inplace=True)
    X = df.drop(columns=['customerid', 'churn'])
    y = df['churn']
    return X, y   

In [43]:
def separate_col_by_dtypes(df:pd.DataFrame) -> list():
    """
    Returns the numerical and categorical feature names as list:
    """
    numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = df.select_dtypes(include=['object', 'string']).columns.tolist()
    return (numerical_features, categorical_features)

In [44]:
def log_confusion_matrix(y_true, y_pred,
                          model_name: str, branch: str):
    fig, ax = plt.subplots(figsize=(6, 5))
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Retained', 'Churned']
    )
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{model_name} ({branch}) — Confusion Matrix')
    plt.tight_layout()
    path = f"cm_{model_name}_{branch}.png"
    plt.savefig(path)
    plt.close()
    mlflow.log_artifact(path)
    os.remove(path)

# 4. Read the data:

In [17]:
clipped_train = pd.read_parquet("Artifacts/feature_engineering/data_clipped/train.parquet")
clipped_test = pd.read_parquet("Artifacts/feature_engineering/data_clipped/test.parquet")

unclipped_train = pd.read_parquet("Artifacts/feature_engineering/data_unclipped/train.parquet")
unclipped_test = pd.read_parquet("Artifacts/feature_engineering/data_unclipped/test.parquet")

## 4.1 Split the data into train and test sets:

In [20]:
# Clipped data:
X_train_clipped, y_train_clipped = create_dep_indep_features(clipped_train)
X_test_clipped, y_test_clipped = create_dep_indep_features(clipped_test)

# Unclipped data:
X_train_unclipped, y_train_unclipped = create_dep_indep_features(unclipped_train)
X_test_unclipped, y_test_unclipped = create_dep_indep_features(unclipped_test)

In [33]:
# The column names will be the same for both the brances(clipped & Unclipped):
numerical_features, categorical_features = separate_col_by_dtypes(X_train_clipped)

## 4.2 Branch Configuration:

In [35]:
branches = {
    'clipped': {
        'X_train': X_train_clipped,
        'y_train': y_train_clipped,
        'X_test': X_test_clipped,
        'y_test': y_test_clipped,
        'numerical_features': numerical_features,
        'categorical_features': categorical_features
    },

    'unclipped': {
        'X_train': X_train_unclipped,
        'y_train': y_train_unclipped,
        'X_test': X_test_unclipped,
        'y_test': y_test_unclipped,
        'numerical_features': numerical_features,
        'categorical_features': categorical_features
    }
}

# 5. Model Configurations:

#### During spot-checking 5 algorithms were picked:

In [38]:
model_configs = {
    
    'SVC': {
        'model': SVC(class_weight='balanced', random_state=42,
                     probability=True),
        'params': {
            'clf__C':      loguniform(0.01, 100),
            'clf__kernel': ['rbf', 'linear'],
            'clf__gamma':  ['scale', 'auto']
        }
    },
    
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'clf__n_estimators':     [100, 200, 300],
            'clf__learning_rate':    loguniform(0.01, 0.3),
            'clf__max_depth':        [3, 4, 5],
            'clf__subsample':        uniform(0.7, 0.3),
            'clf__min_samples_leaf': [10, 20, 30]
        }
    },
    
    'RandomForest': {
        'model': RandomForestClassifier(class_weight='balanced',
                                         random_state=42),
        'params': {
            'clf__n_estimators':     [100, 200, 300],
            'clf__max_depth':        [None, 5, 10, 20],
            'clf__min_samples_leaf': [1, 5, 10, 20],
            'clf__max_features':     ['sqrt', 'log2']
        }
    },
    
    'ExtraTrees': {
        'model': ExtraTreesClassifier(class_weight='balanced',
                                       random_state=42),
        'params': {
            'clf__n_estimators':     [100, 200, 300],
            'clf__max_depth':        [None, 5, 10, 20],
            'clf__min_samples_leaf': [1, 5, 10, 20],
            'clf__max_features':     ['sqrt', 'log2']
        }
    },
    
    'XGBoost': {
    'model': XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42,
        verbosity=0
    ),
    'params': {
        'clf__n_estimators':     [100, 200, 300],
        'clf__max_depth':        [3, 5, 7, 10],
        'clf__learning_rate':    [0.01, 0.05, 0.1, 0.3],
        'clf__subsample':        [0.6, 0.8, 1.0],
        'clf__colsample_bytree': [0.6, 0.8, 1.0],
        'clf__min_child_weight': [1, 3, 5],
        'clf__gamma':            [0, 0.1, 0.3]
    }
}
}

# 6. Build the pre-processors and encoders:

In [47]:
def build_preprocessor():
    preprocessor = ColumnTransformer([
        ('num', RobustScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ], remainder='drop')
    return preprocessor

In [42]:
preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,with_centering,True
,with_scaling,True
,quantile_range,"(25.0, ...)"


# 7. Run the experiments:

In [46]:
def run_experiment(model_name: str, model, param_grid:dict, 
                   branch_name:str, branch_data:dict, 
                   n_iter:int = 30):
    X_train = branch_data['X_train']
    y_train = branch_data['y_train']
    X_test = branch_data['X_test']
    y_test = branch_data['y_test']
    numerical_features = branch_data['numerical_features']
    categorical_features = branch_data['categorical_features']